In [1]:
# 11 setup: imports, functions and modelling tables with claim history (no experiments).
# Run this after a kernel restart. Takes 2-3 minutes.
import sys, time
sys.path.append("..")
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
from src.target import LINES, load_policies, load_claims, build_model_target
from src.features import build_features
from src.history import claim_history
from src.glm import BandedTweedieGLM
from src.evaluate import evaluate
from src.diagnostics import FOLDS

NON_FEATURES = ("row_id", "year", "exposure_clean", "target", "weight")
DROP = {"motor": ["previous_insurer", "customer_tenure"],
        "buildings": ["previous_insurer", "occupation", "region", "customer_tenure"],
        "contents": ["previous_insurer", "occupation", "customer_tenure"]}
HIST = ["hist_1y", "hist_2y"]
GLM_CONFIG = {"motor": dict(alpha=0.5, n_bands=10, drop=None, balance=True),
              "buildings": dict(alpha=0.5, n_bands=20, drop=None, balance=True),
              "contents": dict(alpha=0.3, n_bands=20, drop=None, balance=True)}
PARAMS2 = dict(objective="tweedie", tweedie_variance_power=1.5, metric="tweedie", learning_rate=0.03,
               num_leaves=31, min_data_in_leaf=500, feature_fraction=0.8, bagging_fraction=0.8,
               bagging_freq=1, lambda_l2=1.0, min_data_per_group=1000, cat_smooth=50, cat_l2=10,
               max_cat_threshold=8, verbose=-1, seed=42)
CHOSEN = {"motor": dict(num_leaves=15, min_data_in_leaf=3000),
          "buildings": dict(num_leaves=15, min_data_in_leaf=1000),
          "contents": dict(num_leaves=15, min_data_in_leaf=3000)}
OUT = Path("../results/overnight"); OUT.mkdir(parents=True, exist_ok=True)


def hist_levels(n):
    """Counts -> '0', '1', '2+' ; unknown history stays missing (its own level in the GLM)."""
    return pd.Series(np.select([n == 0, n == 1, n >= 2], ["0", "1", "2+"], default=None), index=n.index,
                     dtype="object").where(n.notna())


def add_history(F, p_rows, p_all_same_line, claims):
    h = claim_history(p_rows, p_all_same_line, claims)
    h["hist_1y"] = hist_levels(h.pol_claims_1y)
    h["hist_2y"] = hist_levels(h.pol_claims_2y)
    out = F.merge(h[["row_id"] + HIST], on="row_id", how="left", validate="one_to_one")
    assert len(out) == len(F)
    return out


def prep2(X, cats=None):
    X = X.copy()
    if cats is None:
        cats = {c: sorted(X[c].dropna().astype(str).unique()) for c in X.columns
                if not pd.api.types.is_numeric_dtype(X[c]) and not pd.api.types.is_bool_dtype(X[c])}
    for c, levels in cats.items():
        s = X[c].astype("string")
        X[c] = pd.Categorical(s.where(s.isin(levels)), categories=levels)
    for c in X.columns:
        if pd.api.types.is_bool_dtype(X[c]):
            X[c] = X[c].astype(int)
    return X, cats


def fit_gbm2(tr, feats, params=PARAMS2, patience=300):
    last = tr.year.max()
    inner, hold = tr[tr.year < last], tr[tr.year == last]
    Xi, cats = prep2(inner[feats]); Xh, _ = prep2(hold[feats], cats)
    di = lgb.Dataset(Xi, inner.target / inner.weight, weight=inner.weight)
    dh = lgb.Dataset(Xh, hold.target / hold.weight, weight=hold.weight, reference=di)
    b = lgb.train(params, di, num_boost_round=5000, valid_sets=[dh],
                  callbacks=[lgb.early_stopping(patience, verbose=False)])
    n_trees = max(int(b.best_iteration * 1.1), 1)
    Xt, cats = prep2(tr[feats])
    model = lgb.train(params, lgb.Dataset(Xt, tr.target / tr.weight, weight=tr.weight), num_boost_round=n_trees)
    k = tr.target.sum() / (model.predict(Xt) * tr.weight).sum()
    return model, cats, k, n_trees


def log(msg):
    line_ = f"{time.strftime('%H:%M:%S')} {msg}"
    print(line_, flush=True)
    with open(OUT / "log.txt", "a") as f:
        f.write(line_ + "\n")


def metrics(va, pred):
    r = evaluate(va.target, pred, va.weight, show=False)
    lf = r["lift"]["A/E"]
    return {"D2": r["D2"], "A/E": r["A/E"], "low": lf.iloc[0], "top": lf.iloc[-1]}


def feats_of(df, line):
    return [c for c in df.columns if c not in NON_FEATURES and c not in DROP[line]]


claims = load_claims()
data = {}
for line in LINES:
    p, pt = load_policies(line, "train"), load_policies(line, "test")
    mt, comp, loading = build_model_target(claims, p, line)
    F = add_history(build_features(line, "train", policies=p), p, [p, pt], claims)
    df = mt.merge(F, on="row_id", how="left", validate="one_to_one")
    data[line] = {"df": df, "loading": loading}
    print(f"{line}: table ready")
print("setup done")

motor: table ready
buildings: table ready
contents: table ready
setup done


In [2]:
# 12 E1-E4: written_premium, customer_service_contacts (leakage), make_model, weather (actual vs average)
# Run after the 11_setup cell (fresh kernel is fine). Rules fixed before results:
#  E1 written_premium: adopt if same-year / next-year association ratio < 1.5 (set at inception) AND D2 >= +0.0005
#  E2 customer_service_contacts: ratio >= 2 -> leakage, excluded whatever the score; else adopt if D2 >= +0.0005
#  E3 make_model (motor): adopt if D2 >= +0.0005 and tails within 0.1 of the base
#  E4 weather: best practical variant (average weather) adopted if it loses < 0.001 D2 vs actual weather;
#     otherwise a team decision
from pathlib import Path
from scipy.stats import spearmanr
from src.features import WEATHER_COLS
from src.target import DATA_DIR

EXTRA_DROP = {"motor": ["soil_type", "roof_type", "energy_rating", "building_age", "property_type", "bedrooms",
                        "flood_zone", "elevation_m", "distance_to_river_km", "discount_applied"],
              "buildings": ["discount_applied"], "contents": ["discount_applied"]}
CAND = ["written_premium", "customer_service_contacts", "make_model"]
cust = pd.read_parquet(DATA_DIR / "customers.parquet")
wthr = pd.read_parquet(DATA_DIR / "weather.parquet")


def raw(line, split="train"):
    """Policy table plus customer columns; tells where each candidate column lives."""
    p = load_policies(line, split)
    extra = [c for c in cust.columns if c not in p.columns or c == "customer_id"]
    return p.merge(cust[extra], on="customer_id", how="left", validate="many_to_one"), p


def tails_ok(c, b):
    return all(abs(c[m] - 1) <= abs(b[m] - 1) + 0.1 for m in ["low", "top"])


report, rows = [], []
for line in LINES:
    df = data[line]["df"]
    r, p = raw(line)
    base = [c for c in feats_of(df, line) if c not in EXTRA_DROP[line]]
    have = [c for c in CAND if c in r.columns and (c != "make_model" or line == "motor")]
    where = {c: ("policy" if c in p.columns else "customer") for c in have}
    print(f"\n===== {line}: candidates found {where}")

    # --- model-free timing check: same-year vs next-year claims (same policy) ---------------------------
    d = df[["row_id", "year", "target"]].merge(r[["row_id", "policy_id"] + have], on="row_id")
    d["claim"] = (d.target > 0).astype(int)
    nxt = d[["policy_id", "year", "claim"]].assign(year=lambda x: x.year - 1).rename(columns={"claim": "claim_next"})
    d = d.merge(nxt, on=["policy_id", "year"], how="left")
    for c in [c for c in have if c != "make_model"]:
        x = d[c] / r.set_index("row_id").loc[d.row_id, "exposure_clean"].to_numpy() if c == "written_premium" else d[c]
        both = d.claim_next.notna()
        cs = spearmanr(x[both], d.claim[both], nan_policy="omit").correlation
        cn = spearmanr(x[both], d.claim_next[both], nan_policy="omit").correlation
        varies = r.groupby("policy_id")[c].nunique().gt(1).mean()
        ratio = cs / cn if cn and abs(cn) > 1e-6 else np.inf
        report.append({"line": line, "feature": c, "table": where[c], "corr_same_year": cs, "corr_next_year": cn,
                       "ratio": ratio, "share_policies_value_changes": varies})
        print(f"  {c}: corr with same-year claim {cs:+.3f}, next-year claim {cn:+.3f}, ratio {ratio:.2f}; "
              f"value changes across years for {varies:.0%} of policies")
    if "make_model" in have:
        g = df[["row_id", "year", "target", "weight"]].merge(r[["row_id", "make_model"]], on="row_id")
        g["period"] = np.where(g.year <= 2022, "early", "late")
        t = g.groupby(["make_model", "period"])[["target", "weight"]].sum()
        rate = (t.target / t.weight).unstack()
        share = g.groupby("make_model").weight.sum() / g.weight.sum()
        rate = rate[share.reindex(rate.index) >= 0.002].dropna()
        print(f"  make_model: {g.make_model.nunique()} levels, {len(rate)} with >= 0.2% weight; "
              f"rank corr early vs late {spearmanr(rate.early, rate.late).correlation:+.2f}")

    # --- validation: base vs + each candidate; weather variants ------------------------------------------
    dfx = df.merge(r[["row_id", "address_id"] + have], on="row_id", how="left", validate="one_to_one")
    if "written_premium" in have:
        dfx["written_premium"] = dfx["written_premium"] / dfx["exposure_clean"]       # premium per year
    variants = {"base (v6 GLM)": base, **{f"+ {c}": base + [c] for c in have}}
    wcols = [c for c in WEATHER_COLS if c in base]
    for vy, tys in FOLDS:
        t0 = time.time()
        tr, va = dfx[dfx.year.isin(tys)], dfx[dfx.year == vy]
        for name, feats in variants.items():
            glm = BandedTweedieGLM(**GLM_CONFIG[line]).fit(tr[feats], tr.target / tr.weight, tr.weight)
            rows.append({"line": line, "year": vy, "model": name,
                         **metrics(va, glm.predict_rate(va[feats]) * va.weight.to_numpy())})
        if wcols:
            avg = wthr[wthr.year.isin(tys)].groupby("address_id")[wcols].mean()          # training years only
            va_avg = va.drop(columns=wcols).merge(avg, left_on="address_id", right_index=True, how="left")
            tr_avg = tr.drop(columns=wcols).merge(avg, left_on="address_id", right_index=True, how="left")
            glm_a = BandedTweedieGLM(**GLM_CONFIG[line]).fit(tr[base], tr.target / tr.weight, tr.weight)
            rows.append({"line": line, "year": vy, "model": "weather: train actual, predict average",
                         **metrics(va_avg, glm_a.predict_rate(va_avg[base]) * va_avg.weight.to_numpy())})
            glm_b = BandedTweedieGLM(**GLM_CONFIG[line]).fit(tr_avg[base], tr_avg.target / tr_avg.weight, tr_avg.weight)
            rows.append({"line": line, "year": vy, "model": "weather: average in both",
                         **metrics(va_avg, glm_b.predict_rate(va_avg[base]) * va_avg.weight.to_numpy())})
        log(f"12 {line} {vy}: {time.time() - t0:.0f}s")

OUT2 = Path("../results/oct20"); OUT2.mkdir(parents=True, exist_ok=True)
rep = pd.DataFrame(report); rep.to_csv(OUT2 / "e1_e2_timing.csv", index=False)
res = pd.DataFrame(rows); res.to_csv(OUT2 / "e1_e4_validation.csv", index=False)
for line in LINES:
    t = res[res.line == line].pivot(index="model", columns="year", values=["D2", "A/E", "low", "top"])
    t.columns = [f"{m}_{y}" for m, y in t.columns]
    t["D2_avg"] = t[["D2_2023", "D2_2024"]].mean(axis=1)
    print(f"\n===== {line} =====")
    print(t.sort_values("D2_avg", ascending=False).round(4).to_string())

print("\nverdicts (rules fixed before results):")
for line in LINES:
    s = res[res.line == line].set_index(["model", "year"])
    b = s.loc["base (v6 GLM)"]
    gain = lambda m: s.loc[m].D2.mean() - b.D2.mean()
    tails = lambda m: all(tails_ok(s.loc[(m, y)], b.loc[y]) for y in b.index)
    for m in [m for m in s.index.get_level_values(0).unique() if m.startswith("+ ")]:
        c = m[2:]
        rr = rep[(rep.line == line) & (rep.feature == c)]
        ratio = rr.ratio.iloc[0] if len(rr) else np.nan
        if c == "customer_service_contacts" and ratio >= 2:
            v = "LEAKAGE -> exclude"
        elif c == "written_premium" and not (ratio < 1.5):
            v = "not shown to be set at inception -> do not adopt"
        else:
            v = "ADOPT" if gain(m) >= 0.0005 and tails(m) else "no"
        print(f"  {line:9s} {m:30s} gain {gain(m):+.4f}  ratio {ratio:.2f}  -> {v}")
    wv = [m for m in s.index.get_level_values(0).unique() if m.startswith("weather")]
    if wv:
        best = max(wv, key=gain)
        print(f"  {line:9s} weather: best practical = '{best}', change vs actual {gain(best):+.4f} -> "
              f"{'use average weather' if gain(best) > -0.001 else 'team decision'}")


===== motor: candidates found {'written_premium': 'policy', 'customer_service_contacts': 'policy', 'make_model': 'policy'}
  written_premium: corr with same-year claim +0.026, next-year claim +0.075, ratio 0.34; value changes across years for 89% of policies
  customer_service_contacts: corr with same-year claim +0.503, next-year claim +0.042, ratio 12.00; value changes across years for 68% of policies
  make_model: 300 levels, 300 with >= 0.2% weight; rank corr early vs late +0.15
14:33:07 12 motor 2023: 96s
14:35:30 12 motor 2024: 143s

===== buildings: candidates found {'written_premium': 'policy', 'customer_service_contacts': 'policy'}
  written_premium: corr with same-year claim +0.009, next-year claim +0.030, ratio 0.29; value changes across years for 89% of policies
  customer_service_contacts: corr with same-year claim +0.332, next-year claim +0.007, ratio 50.75; value changes across years for 64% of policies
14:36:25 12 buildings 2023: 54s
14:37:48 12 buildings 2024: 83s

===

In [3]:
# 13: follow-up to E1-E4. Run in the same kernel as 11_setup (and 12 if you like; not required).
# Purpose
#  A. Test-set checks (no model): how customer_service_contacts looks in test; whether written_premium
#     rises by year (rate changes would shift 2025 rows into higher bands); weather coverage for test addresses.
#  B. Contents uses LightGBM, not the GLM, so E1 (written_premium) and E4 (average weather) are re-tested
#     on the v7 contents GBM (leaves 7, min_data 5000, lambda_l2 50, 3 seeds).
# Rules fixed before results (written Oct 20, before running):
#  - premium: adopted in contents if average D2 >= +0.0005 and tails within 0.1 of the base, as a
#    relative premium (premium per year / that year's average), so rate changes between years do not matter
#  - average weather ("average in both"): adopted if it loses < 0.001 D2 vs actual weather and the tail
#    condition holds; the same definition in every line
from pathlib import Path
from src.features import WEATHER_COLS
from src.target import DATA_DIR

OUT3 = Path("../results/oct20"); OUT3.mkdir(parents=True, exist_ok=True)
wthr = pd.read_parquet(DATA_DIR / "weather.parquet")

# ---------- A. test-set checks ----------
print("A1. customer_service_contacts: train by year vs test")
for line in LINES:
    tr_p, te_p = load_policies(line, "train"), load_policies(line, "test")
    if "customer_service_contacts" not in te_p.columns:
        print(f"  {line}: not in test"); continue
    s = pd.concat([tr_p.assign(year=tr_p.period_start.dt.year)[["year", "customer_service_contacts"]],
                   te_p.assign(year=2025)[["year", "customer_service_contacts"]]])
    print(f"  {line}\n", s.groupby("year").customer_service_contacts
          .agg(mean="mean", share_zero=lambda x: (x == 0).mean(), p90=lambda x: x.quantile(.9)).round(3).to_string())

print("\nA2. written_premium per exposure-year: average by year (rate change check)")
for line in LINES:
    tr_p, te_p = load_policies(line, "train"), load_policies(line, "test")
    s = pd.concat([tr_p.assign(year=tr_p.period_start.dt.year), te_p.assign(year=2025)])
    g = s.groupby("year")[["written_premium", "exposure_clean"]].sum()
    print(f"  {line}: " + "  ".join(f"{y} {v:,.0f}" for y, v in (g.written_premium / g.exposure_clean).items()))

print("\nA3. test addresses with weather in 2021-24")
for line in LINES:
    te_p = load_policies(line, "test")
    if "address_id" in te_p.columns:
        have = te_p.address_id.isin(wthr[wthr.year <= 2024].address_id)
        print(f"  {line}: {have.mean():.2%}")


# ---------- B. contents GBM: premium and average weather ----------
def rel_premium(d):
    """Premium per exposure-year divided by that year's average (exposure-weighted)."""
    pp = d.written_premium / d.exposure_clean
    avg = d.groupby("year").apply(lambda x: x.written_premium.sum() / x.exposure_clean.sum())
    return pp / d.year.map(avg)


line = "contents"
P = dict(PARAMS2, num_leaves=7, min_data_in_leaf=5000, lambda_l2=50)
SEEDS = [42, 43, 44]
df = data[line]["df"]
raw_p = load_policies(line, "train")[["row_id", "address_id", "written_premium"]]
dfx = df.merge(raw_p, on="row_id", how="left", validate="one_to_one")
dfx["prem_rel"] = rel_premium(dfx)
base = [c for c in feats_of(df, line) if c != "discount_applied"]
wcols = [c for c in WEATHER_COLS if c in base]


def gbm_pred(tr, va, feats):
    out = np.zeros(len(va))
    for s in SEEDS:
        m, cats, k, _ = fit_gbm2(tr, feats, params=dict(P, seed=s))
        Xv, _ = prep2(va[feats], cats)
        out += m.predict(Xv) * k / len(SEEDS)
    return out * va.weight.to_numpy()


rows = []
for vy, tys in FOLDS:
    t0 = time.time()
    tr, va = dfx[dfx.year.isin(tys)], dfx[dfx.year == vy]
    avg = wthr[wthr.year.isin(tys)].groupby("address_id")[wcols].mean()
    tr_a = tr.drop(columns=wcols).merge(avg, left_on="address_id", right_index=True, how="left")
    va_a = va.drop(columns=wcols).merge(avg, left_on="address_id", right_index=True, how="left")
    for name, (a, b, f) in {"base (v7 GBM)": (tr, va, base),
                            "+ prem_rel": (tr, va, base + ["prem_rel"]),
                            "weather: average in both": (tr_a, va_a, base),
                            "average weather + prem_rel": (tr_a, va_a, base + ["prem_rel"])}.items():
        rows.append({"year": vy, "model": name, **metrics(b, gbm_pred(a, b, f))})
    log(f"13 contents {vy}: {time.time() - t0:.0f}s")

res = pd.DataFrame(rows); res.to_csv(OUT3 / "13_contents_gbm.csv", index=False)
t = res.pivot(index="model", columns="year", values=["D2", "A/E", "low", "top"])
t.columns = [f"{m}_{y}" for m, y in t.columns]
t["D2_avg"] = t[["D2_2023", "D2_2024"]].mean(axis=1)
print("\n", t.sort_values("D2_avg", ascending=False).round(4).to_string())

s = res.set_index(["model", "year"]); b = s.loc["base (v7 GBM)"]
for m in ["+ prem_rel", "weather: average in both", "average weather + prem_rel"]:
    gain = s.loc[m].D2.mean() - b.D2.mean()
    tails = all(abs(s.loc[(m, y)][c] - 1) <= abs(b.loc[y][c] - 1) + 0.1 for y in b.index for c in ["low", "top"])
    print(f"  {m:28s} gain {gain:+.4f}  tails {'ok' if tails else 'FAIL'}")

A1. customer_service_contacts: train by year vs test
  motor
        mean  share_zero  p90
year                        
2021  0.753       0.583  2.0
2022  0.685       0.617  2.0
2023  0.672       0.622  2.0
2024  0.671       0.622  2.0
2025  0.401       0.670  1.0
  buildings
        mean  share_zero  p90
year                        
2021  0.553       0.616  2.0
2022  0.499       0.650  1.0
2023  0.497       0.653  1.0
2024  0.503       0.651  1.0
2025  0.402       0.669  1.0
  contents
        mean  share_zero  p90
year                        
2021  0.532       0.627  2.0
2022  0.485       0.657  1.0
2023  0.482       0.659  1.0
2024  0.488       0.657  1.0
2025  0.401       0.670  1.0

A2. written_premium per exposure-year: average by year (rate change check)
  motor: 2021 618  2022 701  2023 725  2024 746  2025 766
  buildings: 2021 328  2022 385  2023 405  2024 427  2025 449
  contents: 2021 141  2022 165  2023 174  2024 183  2025 193

A3. test addresses with weather in 2021-24
  m

In [5]:
# 14: v8 submission. Run in the same kernel after 11_setup. About 10 minutes.
# Purpose
#  1. Rebuild v7 from code (actual weather, no premium) and compare with the v7.csv we submitted:
#     a reproducibility check — if the rebuild does not match, stop and look before submitting v8.
#  2. Build v8 = v7 + adopted Oct 20 changes:
#       all lines: average weather per address (2021-24) in training and prediction  [E4, notebook 12/13]
#       contents : relative premium (premium per year / that year's average)          [E1, notebook 13]
# Expected change, written before submitting (Oct 20):
#   validation: motor -0.0005, buildings +0.0002, contents +0.0013 (each line's own D2)
#   leaderboard (all lines together): about -0.0003 to +0.0005, i.e. within the +-0.001 noise range.
#   The main reason for v8 is that it does not use 2025 weather, which is not known when pricing.
from pathlib import Path
from src.target import DATA_DIR

SUB = Path("../submissions"); SUB.mkdir(exist_ok=True)
EXTRA_DROP = {"motor": ["soil_type", "roof_type", "energy_rating", "building_age", "property_type", "bedrooms",
                        "flood_zone", "elevation_m", "distance_to_river_km", "discount_applied"],
              "buildings": ["discount_applied"], "contents": ["discount_applied"]}
PARETO_LOADING = 2.83                                   # motor, per weight (notebook 11h)
GBM_P = dict(PARAMS2, num_leaves=7, min_data_in_leaf=5000, lambda_l2=50)
SEEDS = [42, 43, 44]


def rel_premium(d):
    pp = d.written_premium / d.exposure_clean
    avg = d.groupby("year").written_premium.sum() / d.groupby("year").exposure_clean.sum()
    return pp / d.year.map(avg)


def tables(line, weather):
    """Train modelling table and test feature table for one weather mode."""
    p, pt = load_policies(line, "train"), load_policies(line, "test")
    mt = data[line]["df"][["row_id", "year", "exposure_clean", "target", "weight"]]
    F = add_history(build_features(line, "train", policies=p, weather_mode=weather), p, [p, pt], claims)
    Ft = add_history(build_features(line, "test", policies=pt, weather_mode=weather), pt, [p, pt], claims)
    tr = mt.merge(F, on="row_id", how="left", validate="one_to_one")
    te = pt[["row_id", "exposure_clean", "written_premium"]].assign(year=2025).merge(
        Ft, on="row_id", how="left", validate="one_to_one")
    tr = tr.merge(p[["row_id", "written_premium"]], on="row_id", how="left", validate="one_to_one")
    tr["prem_rel"], te["prem_rel"] = rel_premium(tr), rel_premium(te)
    return tr, te


def predict_line(line, weather, premium):
    tr, te = tables(line, weather)
    feats = [c for c in feats_of(data[line]["df"], line) if c not in EXTRA_DROP[line]]
    if premium:
        feats = feats + ["prem_rel"]
    if line == "contents":
        rate = np.zeros(len(te))
        for s in SEEDS:
            m, cats, k, _ = fit_gbm2(tr, feats, params=dict(GBM_P, seed=s))
            Xt, _ = prep2(te[feats], cats)
            rate += m.predict(Xt) * k / len(SEEDS)
        extra = "LightGBM, 3 seeds"
    else:
        glm = BandedTweedieGLM(**GLM_CONFIG[line]).fit(tr[feats], tr.target / tr.weight, tr.weight)
        rate = glm.predict_rate(te[feats])
        if line == "motor":                               # Pareto loading, allocated by risk
            f = 1 + PARETO_LOADING / (tr.target.sum() / tr.weight.sum())
            rate = rate * f
            extra = f"GLM, k={glm.k_:.4f}, loading factor {f:.4f}"
        else:                                             # buildings: observed loading, flat
            rate = rate + data[line]["loading"]
            extra = f"GLM, k={glm.k_:.4f}, flat loading {data[line]['loading']:.2f}"
    pred = np.maximum(rate * te.exposure_clean.to_numpy(), 0)
    print(f"  {line}: {extra}; mean rate {np.average(rate, weights=te.exposure_clean):.2f}, "
          f"total {pred.sum():,.0f}")
    return pd.DataFrame({"row_id": te.row_id, "pred": pred})


sample = pd.read_parquet(DATA_DIR / "sample_submission.parquet")
id_col, val_col = sample.columns[:2]


def build(name, weather, premium_lines):
    print(f"\n{name}")
    out = pd.concat([predict_line(l, weather, l in premium_lines) for l in LINES])
    sub = sample[[id_col]].merge(out.rename(columns={"row_id": id_col, "pred": val_col}), on=id_col, how="left")
    assert sub[val_col].notna().all() and len(sub) == len(sample)
    return sub


# 1. reproducibility check
v7r = build("v7 rebuilt (actual weather, no premium)", "actual", [])
old = SUB / "v7.csv"
if old.exists():
    v7 = pd.read_csv(old).merge(v7r, on=id_col, suffixes=("_v7", "_rebuilt"))
    a, b = v7[f"{val_col}_v7"], v7[f"{val_col}_rebuilt"]
    print(f"v7 vs rebuilt: totals {a.sum():,.0f} vs {b.sum():,.0f} ({b.sum() / a.sum() - 1:+.3%}); "
          f"correlation {np.corrcoef(a, b)[0, 1]:.5f}; max abs ratio diff {(b / a.clip(lower=1e-9) - 1).abs().max():.3%}")
else:
    print(f"{old} not found: put v7.csv there to compare (the check is the point of step 1)")

# 2. v8
v8 = build("v8 (average weather; contents relative premium)", "average", ["contents"])
v8.to_csv(SUB / "v8.csv", index=False)
cmp = v7r.merge(v8, on=id_col, suffixes=("_v7", "_v8"))
print(f"\nv8 vs v7 rebuilt: total {cmp[f'{val_col}_v8'].sum() / cmp[f'{val_col}_v7'].sum() - 1:+.2%}, "
      f"correlation {np.corrcoef(cmp[f'{val_col}_v7'], cmp[f'{val_col}_v8'])[0, 1]:.4f}")
print(f"saved {SUB / 'v8.csv'}")


v7 rebuilt (actual weather, no premium)
  motor: GLM, k=1.0189, loading factor 1.0054; mean rate 495.52, total 99,095,048
  buildings: GLM, k=1.0401, flat loading 2.33; mean rate 479.28, total 72,008,798
  contents: LightGBM, 3 seeds; mean rate 109.67, total 16,464,103
v7 vs rebuilt: totals 187,570,856 vs 187,567,949 (-0.002%); correlation 0.99996; max abs ratio diff 19.123%

v8 (average weather; contents relative premium)
  motor: GLM, k=1.0189, loading factor 1.0054; mean rate 508.32, total 101,654,845
  buildings: GLM, k=1.0407, flat loading 2.33; mean rate 485.93, total 73,007,884
  contents: LightGBM, 3 seeds; mean rate 108.63, total 16,308,227

v8 vs v7 rebuilt: total +1.81%, correlation 0.9863
saved ../submissions/v8.csv


In [6]:
# 2025 weather vs 2021-24: is the test year outside the training range?
w = wthr.copy()
print(w.groupby("year")[WEATHER_COLS].mean().round(2).to_string())
past = w[w.year <= 2024]
for c in WEATHER_COLS:
    lo, hi = past[c].quantile([0.01, 0.99])
    out = ((w.loc[w.year == 2025, c] < lo) | (w.loc[w.year == 2025, c] > hi)).mean()
    print(f"{c:22s} share of 2025 values outside the 2021-24 1-99% range: {out:.1%}")

# attribution experiment: average weather only, no premium
v8a = build("v8a (average weather only)", "average", [])
v8a.to_csv(SUB / "v8a_experiment.csv", index=False)

      hail_days  wind_exposure_index  freezing_days  dry_summer_index  wildfire_index  rainfall_mm  mean_temperature_c
year                                                                                                                  
2021       5.64                 4.60          41.27              0.49            0.18      1143.82               12.22
2022       6.72                 4.51          36.98              0.47            0.15      1023.51               12.43
2023       6.26                 4.39          34.70              0.55            0.13       980.69               13.08
2024       6.77                 4.35          39.48              0.59            0.16      1036.39               12.97
2025       5.89                 4.88          35.89              0.49            0.17      1039.97               12.62
hail_days              share of 2025 values outside the 2021-24 1-99% range: 0.3%
wind_exposure_index    share of 2025 values outside the 2021-24 1-99% range: 3.7%
fre

In [7]:
# 15 E5: bodily injury trend + reform, and truncation-corrected completeness (Fable review A1, C5)
# Run in the same kernel after 11_setup (13/14 not needed). About 8-10 minutes.
# Purpose
#  Part 1  Completeness: the current lag pattern uses 2021 losses only, normalised by what was reported
#          by 2024-12-30, so lags longer than the time observed are missing (right truncation).
#          Re-estimate the lag distribution with all 2021-24 losses using the Lynden-Bell estimator,
#          which accounts for each claim's own observation window. Also check whether BI reporting
#          sped up after 2023.
#  Part 2  BI severity: estimate the pre-reform trend, post-reform trend and reform step jointly, on
#          equal-maturity claims (reported within L days, cohorts fully observed to L), with a Gamma
#          GLM (mean, not median). Check the step by loss vs report quarter and on closed claims only,
#          and the change by claim size.
#  Part 3  Motor target variants with average weather (v8 basis): current / new completeness / new BI /
#          both / peril-level grossing; validation, yearly levels and the 2025 total.
# Rules fixed before results (Oct 20):
#  R1 Lynden-Bell completeness replaces the 2021 pattern if it is monotone and non-BI perils are
#     >= 0.95 complete for 2021 (a correction of a known statistical bias, not a modelling choice).
#  R2 BI parameters come from the L=180 Gamma fit (the only cohort that sees post-reform claims). Adopted if
#     (a) its pre-reform trend is within 3 points of the L=365 and L=730 fits, (b) the step shows by loss
#     quarter in closed claims too, (c) with both fixes, validation A/E moves toward 1 or stays within
#     0.02 of 1 in both years, and the spread of yearly levels does not grow.
#  R3 Size-dependent reform is recorded, and implemented later only if the top-decile ratio differs from
#     the median ratio by more than 0.1.
#  R4 Peril-level grossing replaces the line-level weight if D2 >= +0.0005 on the same validation actuals
#     and tails within 0.1.
# Note: average weather here uses 2021-24 (including the validation year) for speed; it is the same for
# all variants, so comparisons between variants are fair.
from sklearn.linear_model import GammaRegressor
from src.target import ADJ, YEARS, VALUATION_DATE, match_claims, adjust_claims, completeness

LINE = "motor"
CAP = ADJ[LINE]["cap"]
EXTRA_DROP_M = ["soil_type", "roof_type", "energy_rating", "building_age", "property_type", "bedrooms",
                "flood_zone", "elevation_m", "distance_to_river_km", "discount_applied"]
p, pt = load_policies(LINE, "train"), load_policies(LINE, "test")
m = match_claims(claims, p, LINE, True, cap=None)
m = m[m.loss_year.between(2021, 2024)].copy()
m["trunc"] = (VALUATION_DATE - m.loss_date).dt.days


# ---------------- Part 1: completeness ----------------
def lynden_bell(lag, trunc):
    """Report-lag CDF under right truncation (a claim is seen only if lag <= its own window).
    F(t) = prod_{s > t} (1 - d(s) / R(s)),  R(s) = #{lag <= s <= window}."""
    lag, trunc = np.asarray(lag), np.asarray(trunc)
    s, d = np.unique(lag, return_counts=True)
    R = np.searchsorted(np.sort(lag), s, side="right") - np.searchsorted(np.sort(trunc), s, side="left")
    logq = np.log(np.clip(1 - d / R, 1e-12, 1))
    tail = np.concatenate([np.cumsum(logq[::-1])[::-1][1:], [0.0]])     # sum over s_j > s_k
    def F(t):
        k = np.searchsorted(s, t, side="right") - 1
        return np.where(k >= 0, np.exp(tail[np.clip(k, 0, None)]), 0.0)
    return F


LB = {per: lynden_bell(g.report_lag, g.trunc) for per, g in m.groupby("peril")}


_CACHE = {}


def comp_lb(peril, year):
    if (peril, year) in _CACHE:
        return _CACHE[(peril, year)]
    ages = (VALUATION_DATE - pd.date_range(f"{year}-01-01", f"{year}-12-31")).days.to_numpy()
    _CACHE[(peril, year)] = float(LB[peril](ages).mean())
    return _CACHE[(peril, year)]


tab = pd.DataFrame([{"peril": per, "year": y, "current": completeness(m, per, y), "lynden_bell": comp_lb(per, y)}
                    for per in sorted(LB) for y in YEARS])
print("Part 1. completeness by peril and year: current (2021 pattern) vs Lynden-Bell (all years)")
print(tab.pivot(index="peril", columns="year", values=["current", "lynden_bell"]).round(3).to_string())
ok_mono = all(np.all(np.diff(LB[per](np.arange(0, 1500))) >= -1e-12) for per in LB)
ok_2021 = tab[(tab.year == 2021) & (tab.peril != "bodily_injury")].lynden_bell.min()
print(f"R1: monotone {ok_mono}; lowest non-BI 2021 completeness {ok_2021:.3f} -> "
      f"{'ADOPT' if ok_mono and ok_2021 >= 0.95 else 'do not adopt'}")

bi = m[m.peril == "bodily_injury"].copy()
h1 = bi[bi.loss_date.dt.month <= 6]
print("\nBI reporting speed (Jan-Jun losses reported within 365 days): share reported within ...")
for y in [2021, 2022, 2023]:
    s = h1[(h1.loss_year == y) & (h1.report_lag <= 365)].report_lag
    print(f"  {y}: 30d {(s <= 30).mean():.1%}  90d {(s <= 90).mean():.1%}  180d {(s <= 180).mean():.1%}  (n={len(s)})")


# ---------------- Part 2: BI severity ----------------
bi["sev"] = np.minimum(bi.incurred_to_date, CAP)
bi["t"] = 2021 + (bi.loss_date - pd.Timestamp("2021-01-01")).dt.days / 365.25


def gamma_fit(d, with_post_slope=True):
    d = d[d.sev > 0]                                    # severity of claims with a cost
    X = pd.DataFrame({"pre_t": np.minimum(d.t, 2023) - 2021, "post": (d.t >= 2023).astype(float)})
    if with_post_slope:
        X["post_t"] = np.maximum(d.t - 2023, 0)
    if d.t.max() < 2023:
        X = X[["pre_t"]]
    g = GammaRegressor(alpha=0, max_iter=2000).fit(X, d.sev)
    b = dict(zip(X.columns, g.coef_))
    return {"n": len(d), "g_pre": np.exp(b["pre_t"]) - 1, "step": np.exp(b.get("post", np.nan)),
            "g_post": np.exp(b.get("post_t", np.nan)) - 1}


fits = {}
for L in [180, 365, 730]:
    d = bi[(bi.report_lag <= L) & (bi.loss_date <= VALUATION_DATE - pd.Timedelta(days=L))]
    fits[f"L={L}"] = gamma_fit(d, with_post_slope=(L == 180))
d180 = bi[(bi.report_lag <= 180) & (bi.loss_date <= VALUATION_DATE - pd.Timedelta(days=180))]
fits["L=180 closed only"] = gamma_fit(d180[d180.status == "closed"])
ft = pd.DataFrame(fits).T
print("\nPart 2. BI severity, Gamma GLM (log link) on gross capped incurred")
print(ft.round(4).to_string())

print("\nBI mean severity (L=180 cohort) by loss quarter vs report quarter, and closed only by loss quarter")
d180 = d180.assign(lq=d180.loss_date.dt.to_period("Q"), rq=d180.report_date.dt.to_period("Q"))
q = pd.DataFrame({"by_loss_q": d180.groupby("lq").sev.mean(), "by_report_q": d180.groupby("rq").sev.mean(),
                  "closed_by_loss_q": d180[d180.status == "closed"].groupby("lq").sev.mean()})
print(q.loc["2022Q1":"2024Q2"].round(0).to_string())

d365 = bi[(bi.report_lag <= 365) & (bi.loss_date <= VALUATION_DATE - pd.Timedelta(days=365))]
qs = [0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
a, b = d365[d365.loss_year == 2022].sev.quantile(qs), d365[d365.loss_year == 2023].sev.quantile(qs)
g365 = fits["L=365"]["g_pre"]
size = pd.DataFrame({"2022": a.values, "2023": b.values, "ratio": (b / a).values,
                     "ratio net of 1y trend": (b / a).values / (1 + g365)}, index=qs)
print("\nBI severity by size, 2023 vs 2022 losses (reported within 365 days)")
print(size.round(3).to_string())
spread = size.loc[0.9, "ratio net of 1y trend"] - size.loc[0.5, "ratio net of 1y trend"]
print(f"R3: top decile minus median ratio {spread:+.3f} -> "
      f"{'size-dependent reform (later)' if abs(spread) > 0.1 else 'flat factor acceptable'}")

c = fits["L=180"]
pre_ok = all(abs(c["g_pre"] - fits[k]["g_pre"]) <= 0.03 for k in ["L=365", "L=730"])
print(f"R2(a) pre-trend stable across cohorts: {pre_ok}  (L=180 {c['g_pre']:+.3f}, "
      f"L=365 {fits['L=365']['g_pre']:+.3f}, L=730 {fits['L=730']['g_pre']:+.3f})")
print(f"R2(b) closed-only step {fits['L=180 closed only']['step']:.3f} vs all {c['step']:.3f} (check visually above too)")


# ---------------- Part 3: target variants ----------------
def bi_mult(y, g_pre, step, g_post):
    """Factor bringing a BI loss of year y to the 2025 level (years measured mid-year)."""
    if y >= 2023:
        return (1 + g_post) ** (2025 - y)
    return (1 + g_pre) ** (2022.5 - y) * step * (1 + g_post) ** 2.5


def adjusted(new_bi):
    a = adjust_claims(m, ADJ[LINE])
    if new_bi:
        b = a.peril == "bodily_injury"
        f = a.loc[b, "loss_year"].map(lambda y: bi_mult(y, c["g_pre"], c["step"], c["g_post"]))
        net = np.maximum(np.maximum(a.loc[b, "incurred_to_date"] * f - a.loc[b, "excess"], 0)
                         - a.loc[b, "recoveries"] * f, 0)
        a.loc[b, "net_adj_uncapped"], a.loc[b, "net_adj"] = net, np.minimum(net, CAP)
    return a


def line_comp(a, comp_fn):
    out = {}
    for y in YEARS:
        r = a[a.loss_year == y]
        ult = sum(r.loc[r.peril == per, "net_adj"].sum() / comp_fn(per, y) for per in r.peril.unique())
        out[y] = r.net_adj.sum() / ult
    return out


def variant(new_bi, comp_kind):
    a = adjusted(new_bi)
    df = p[["row_id", "year", "exposure_clean"]].copy()
    if comp_kind == "peril_gross":
        a = a.assign(g=a.net_adj / [comp_lb(per, y) for per, y in zip(a.peril, a.loss_year)])
        df["target"] = df.row_id.map(a.groupby("row_id").g.sum()).fillna(0.0)
        df["weight"] = df.exposure_clean
        comp = None
    else:
        fn = (lambda per, y: completeness(m, per, y)) if comp_kind == "current" else comp_lb
        comp = line_comp(a, fn)
        df["target"] = df.row_id.map(a.groupby("row_id").net_adj.sum()).fillna(0.0)
        df["weight"] = df.exposure_clean * df.year.map(comp)
    return df, comp


F = add_history(build_features(LINE, "train", policies=p, weather_mode="average"), p, [p, pt], claims)
Ft = add_history(build_features(LINE, "test", policies=pt, weather_mode="average"), pt, [p, pt], claims)
feats = [x for x in feats_of(data[LINE]["df"], LINE) if x not in EXTRA_DROP_M]
VARIANTS = {"V0 current": (False, "current"), "V1 new completeness": (False, "lb"),
            "V2 new BI": (True, "current"), "V3 both": (True, "lb"), "V4 both, peril grossing": (True, "peril_gross")}
res, levels, totals, frames = [], {}, {}, {}
for name, (nb, ck) in VARIANTS.items():
    t0 = time.time()
    df, comp = variant(nb, ck)
    df = df.merge(F, on="row_id", how="left", validate="one_to_one")
    frames[name] = df
    g = df.groupby("year")[["target", "weight"]].sum()
    levels[name] = (g.target / g.weight).round(1).to_dict()
    for vy, tys in FOLDS:
        tr = df[df.year.isin(tys)]
        va = frames["V3 both"][frames["V3 both"].year == vy] if name.startswith("V4") else df[df.year == vy]
        glm = BandedTweedieGLM(**GLM_CONFIG[LINE]).fit(tr[feats], tr.target / tr.weight, tr.weight)
        res.append({"variant": name, "year": vy, **metrics(va, glm.predict_rate(va[feats]) * va.weight.to_numpy())})
    glm = BandedTweedieGLM(**GLM_CONFIG[LINE]).fit(df[feats], df.target / df.weight, df.weight)
    totals[name] = float((glm.predict_rate(Ft[feats]) * pt.set_index("row_id").loc[Ft.row_id, "exposure_clean"].to_numpy()).sum())
    log(f"15 {name}: {time.time() - t0:.0f}s; completeness {comp and {k: round(v, 3) for k, v in comp.items()}}")

chk = frames["V0 current"].target.sum() / data[LINE]["df"].target.sum() - 1
print(f"check: V0 target total vs setup target {chk:+.4%} (should be ~0)")
r = pd.DataFrame(res); r.to_csv("../results/oct20/15_e5_variants.csv", index=False)
t = r.pivot(index="variant", columns="year", values=["D2", "A/E", "low", "top"])
t.columns = [f"{a}_{b}" for a, b in t.columns]
t["D2_avg"] = t[["D2_2023", "D2_2024"]].mean(axis=1)
lv = pd.DataFrame(levels).T
lv["spread"] = lv.max(axis=1) / lv.min(axis=1) - 1
tot = pd.Series(totals); tot = pd.DataFrame({"2025 motor total": tot.round(0), "vs V0": (tot / tot.iloc[0] - 1).round(4)})
print("\nPart 3. validation (V4 is evaluated on V3's actuals, so V3 vs V4 is like for like)")
print(t.round(4).to_string())
print("\n2025-level cost per weight by year, and spread")
print(lv.round(3).to_string())
print("\n2025 motor prediction (before loading)")
print(tot.to_string())

ae0 = t.loc["V0 current"]; ae3 = t.loc["V3 both"]
toward = all((abs(ae3[f"A/E_{y}"] - 1) <= abs(ae0[f"A/E_{y}"] - 1)) or abs(ae3[f"A/E_{y}"] - 1) <= 0.02 for y in [2023, 2024])
print(f"\nR2(c) V3 A/E toward 1 or within 0.02 in both years: {toward}; "
      f"spread V0 {lv.loc['V0 current', 'spread']:.3f} vs V3 {lv.loc['V3 both', 'spread']:.3f}")
g4 = t.loc["V4 both, peril grossing", "D2_avg"] - t.loc["V3 both", "D2_avg"]
print(f"R4 peril grossing vs V3: D2 {g4:+.4f}")

Part 1. completeness by peril and year: current (2021 pattern) vs Lynden-Bell (all years)
                 current                      lynden_bell                     
year                2021   2022   2023   2024        2021   2022   2023   2024
peril                                                                         
bodily_injury      0.999  0.993  0.942  0.560       0.998  0.991  0.941  0.562
glass_hail         1.000  1.000  0.998  0.970       1.000  1.000  0.998  0.971
own_damage         1.000  1.000  0.999  0.965       1.000  1.000  0.999  0.965
theft              1.000  1.000  0.998  0.977       1.000  1.000  0.999  0.979
third_party_prop   1.000  1.000  0.998  0.935       1.000  1.000  0.999  0.937
R1: monotone True; lowest non-BI 2021 completeness 1.000 -> ADOPT

BI reporting speed (Jan-Jun losses reported within 365 days): share reported within ...
  2021: 30d 17.5%  90d 44.3%  180d 72.5%  (n=862)
  2022: 30d 19.6%  90d 44.9%  180d 71.4%  (n=898)
  2023: 30d 17.8%  90d 

In [8]:
# 16: step-change scan — find sudden level changes (law, product, claims handling) in every series.
# Run in the same kernel after 11_setup. About 2-4 minutes. Reusable as-is on the Part 2 data.
# Purpose
#  For every line x peril, build monthly series and test "trend + one step" against "trend only":
#   - frequency by loss month (claims reported within 180 days, per exposure)
#   - severity by loss month, by report month and by policy-start month (the basis where a step is
#     strongest points to the cause: loss date = law / environment, report date = claims handling or
#     reserving, policy start = product or underwriting)
#   - severity on closed claims only (a step in reserves only would vanish here)
#   - repudiation rate by report month (a handling change can look like a frequency drop)
#  and, on the policy side, the monthly average of every numeric column and the share of each level of
#  small categorical columns, by policy-start month (e.g. a jump in compulsory excess).
# Equal maturity: only claims reported within 180 days of loss, loss months up to 2024-06.
# Method: log of the series; weighted least squares with trend (+ month-of-year effects for claim series);
#  best single break chosen by least squares; dBIC = n ln(SSE0/SSE1) - 2 ln(n) (a break costs a level and a date).
# Rules fixed before results (Oct 20):
#  S0 positive controls: the scan must find the known BI severity step (2023, loss date, about 0.8) and the
#     own damage / third-party property frequency steps (2023). If not, the scan is not trusted.
#  S1 a claim series is flagged if dBIC >= 10 and the step is at least 5%; a policy series if dBIC >= 10
#     and at least 3%.
#  S2 a flagged step not already in the target is investigated, and adjusted only if it is strongest on the
#     loss-date (or policy-start) basis, the level after the break is flat, and validation A/E moves toward
#     1 in both years. The scan itself changes nothing.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from src.target import VALUATION_DATE, match_claims, load_claims

OUT4 = Path("../results/oct20"); OUT4.mkdir(parents=True, exist_ok=True)
L = 180
END = (VALUATION_DATE - pd.Timedelta(days=L)).to_period("M") - 1
MONTHS = pd.period_range("2021-01", END, freq="M")
PMONTHS = pd.period_range("2021-01", "2024-12", freq="M")
MIN_COUNT = 20                      # skip claim series with a median monthly count below this


def monthly_exposure(p, months):
    s, e = p.period_start.to_numpy(), p.period_end.to_numpy()
    scale = (p.exposure_clean / p.days).to_numpy()
    out = []
    for mth in months:
        ms, me = np.datetime64(mth.start_time.date()), np.datetime64(mth.end_time.date())
        ov = (np.minimum(e, me) - np.maximum(s, ms)).astype("timedelta64[D]").astype(int) + 1
        out.append((np.clip(ov, 0, None) * scale).sum())
    return pd.Series(out, index=months)


def step_test(y, w=None, seasonal=False, min_seg=6):
    y = np.asarray(y, float); T = len(y); t = np.arange(T)
    w = np.ones(T) if w is None else np.nan_to_num(np.asarray(w, float))
    ok = np.isfinite(y) & (y > 0) & (w > 0)
    ly = np.log(np.where(ok, y, 1.0))
    cols = [np.ones(T), t / 12]
    if seasonal:
        cols += [(t % 12 == j).astype(float) for j in range(1, 12)]
    X0 = np.column_stack(cols)

    def fit(X):
        sw = np.sqrt(w[ok]); b, *_ = np.linalg.lstsq(X[ok] * sw[:, None], ly[ok] * sw, rcond=None)
        return (((ly[ok] - X[ok] @ b) * sw) ** 2).sum(), b

    s0, b0 = fit(X0)
    best = None
    for k in range(min_seg, T - min_seg + 1):
        s1, b1 = fit(np.column_stack([X0, (t >= k).astype(float)]))
        if best is None or s1 < best[0]:
            best = (s1, k, b1)
    n = ok.sum()
    return {"k": best[1], "step": float(np.exp(best[2][-1])), "trend_pa": float(np.exp(best[2][1]) - 1),
            "dbic": float(n * np.log(s0 / best[0]) - 2 * np.log(n)), "n": int(n)}


rows, store = [], {}


def add(line, peril, measure, basis, y, w, months, seasonal):
    y = pd.Series(y).reindex(months); w = pd.Series(w).reindex(months).fillna(0)
    if measure != "policy" and w.median() < MIN_COUNT and measure not in ("repudiation rate",):
        return
    r = step_test(y.to_numpy(), w.to_numpy(), seasonal)
    r.update(line=line, peril=peril, measure=measure, basis=basis, brk=str(months[r["k"]]))
    rows.append(r); store[(line, peril, measure, basis)] = (y, months[r["k"]])


claims_all = load_claims(drop_repudiated=False)
for line in LINES:
    t0 = time.time()
    p = load_policies(line, "train")
    expo = monthly_exposure(p, MONTHS)
    m = match_claims(claims_all, p, line, True, cap=None)
    m = m.assign(lm=m.loss_date.dt.to_period("M"), rm=m.report_date.dt.to_period("M"),
                 pm=m.period_start.dt.to_period("M"))
    mat = m[(m.report_lag <= L) & (m.lm >= MONTHS[0]) & (m.lm <= END)]
    for per, g in mat.groupby("peril"):
        paid = g[g.status != "repudiated"]
        cnt = paid.groupby("lm").size().reindex(MONTHS, fill_value=0)
        add(line, per, "frequency", "loss date", cnt / expo, cnt, MONTHS, True)
        cap = paid.incurred_to_date.quantile(0.995)
        s = paid[paid.incurred_to_date > 0].assign(sev=lambda d: d.incurred_to_date.clip(upper=cap))
        for basis, col in [("loss date", "lm"), ("report date", "rm"), ("policy start", "pm")]:
            a = s.groupby(col).sev.agg(["mean", "size"])
            add(line, per, "severity", basis, a["mean"], a["size"], MONTHS, True)
        a = s[s.status == "closed"].groupby("lm").sev.agg(["mean", "size"])
        add(line, per, "severity (closed)", "loss date", a["mean"], a["size"], MONTHS, True)
        rr = g.groupby("rm").status.agg(lambda x: (x == "repudiated").mean())
        nrep = g.groupby("rm").status.agg(lambda x: (x == "repudiated").sum())
        if nrep.sum() >= 50:
            add(line, per, "repudiation rate", "report date", rr, nrep, MONTHS, False)
    # policy side
    pm = p.period_start.dt.to_period("M")
    skip = {"exposure", "days", "exp_dates", "exposure_clean", "year"}
    pp = p.assign(premium_per_year=p.written_premium / p.exposure_clean) if "written_premium" in p else p
    for c in pp.columns:
        if c in skip or c.endswith("_id") or c == "row_id":
            continue
        col = pp[c]
        if pd.api.types.is_bool_dtype(col):
            add(line, "-", "policy", f"share {c}", col.groupby(pm).mean(), col.groupby(pm).size(), PMONTHS, False)
        elif pd.api.types.is_numeric_dtype(col):
            mean = col.groupby(pm).mean()
            if (mean > 0).all():
                add(line, "-", "policy", f"mean {c}", mean, col.groupby(pm).size(), PMONTHS, False)
        elif 2 <= col.nunique() <= 8:
            for lev in col.dropna().unique():
                sh = (col == lev).groupby(pm).mean()
                if (sh > 0).all():
                    add(line, "-", "policy", f"share {c}={lev}", sh, col.groupby(pm).size(), PMONTHS, False)
    log(f"16 {line}: {time.time() - t0:.0f}s")

res = pd.DataFrame(rows)
res["size"] = (res.step - 1).abs()
res.to_csv(OUT4 / "16_step_scan.csv", index=False)
show = ["line", "peril", "measure", "basis", "brk", "step", "trend_pa", "dbic", "n"]

print("S0 positive controls")
for line, per, meas in [("motor", "bodily_injury", "severity"), ("motor", "own_damage", "frequency"),
                        ("motor", "third_party_prop", "frequency")]:
    print(res[(res.line == line) & (res.peril == per) & (res.measure == meas) & (res.basis == "loss date")][show]
          .round(3).to_string(header=False, index=False))

cl = res[res.measure != "policy"]
flag_c = cl[(cl.dbic >= 10) & (cl["size"] >= 0.05)].sort_values("dbic", ascending=False)
po = res[res.measure == "policy"]
flag_p = po[(po.dbic >= 10) & (po["size"] >= 0.03)].sort_values("dbic", ascending=False)
print(f"\nS1 flagged claim series ({len(flag_c)} of {len(cl)})")
print(flag_c[show].round(3).to_string(index=False))
print(f"\nS1 flagged policy series ({len(flag_p)} of {len(po)})")
print(flag_p[show].round(3).to_string(index=False))

print("\nSeverity: basis with the strongest step (loss date = law/environment, report date = handling, "
      "policy start = product)")
sv = res[res.measure == "severity"]
best = sv.loc[sv.groupby(["line", "peril"]).dbic.idxmax()]
print(best[best.dbic >= 10][show].round(3).to_string(index=False))

# figure: top flagged series
top = pd.concat([flag_c.head(9), flag_p.head(3)])
if len(top):
    n = len(top); fig, axes = plt.subplots((n + 2) // 3, 3, figsize=(12, 2.6 * ((n + 2) // 3)), squeeze=False)
    for ax, (_, r) in zip(axes.flat, top.iterrows()):
        y, b = store[(r.line, r.peril, r.measure, r.basis)]
        ax.plot(y.index.to_timestamp(), y.values, lw=1)
        ax.axvline(b.to_timestamp(), color="r", ls="--", lw=1)
        ax.set_title(f"{r.line} {r.peril} {r.measure}\n{r.basis}: step x{r.step:.2f} at {r.brk}", fontsize=8)
        ax.tick_params(labelsize=7)
    for ax in axes.flat[n:]:
        ax.axis("off")
    fig.tight_layout(); fig.savefig(OUT4 / "16_step_scan.png", dpi=120)
    print(f"\nfigure saved: {OUT4 / '16_step_scan.png'}")

/var/folders/62/vbb1n0sn0z36rw_dx0zhb7_80000gn/T/ipykernel_20157/405053739.py:71: RuntimeWarning: divide by zero encountered in scalar divide
  "dbic": float(n * np.log(s0 / best[0]) - 2 * np.log(n)), "n": int(n)}


16:01:23 16 motor: 2s
16:01:23 16 buildings: 1s


/var/folders/62/vbb1n0sn0z36rw_dx0zhb7_80000gn/T/ipykernel_20157/405053739.py:71: RuntimeWarning: divide by zero encountered in scalar divide
  "dbic": float(n * np.log(s0 / best[0]) - 2 * np.log(n)), "n": int(n)}


16:01:24 16 contents: 1s
S0 positive controls
motor bodily_injury severity loss date 2023-01 0.806 0.034 4.836 42
motor own_damage frequency loss date 2023-01 0.918 0.006 10.021 42
motor third_party_prop frequency loss date 2022-01 1.06 -0.044 2.541 42

S1 flagged claim series (2 of 68)
     line      peril   measure       basis     brk  step  trend_pa   dbic  n
buildings subsidence  severity report date 2022-09 0.598     0.353 13.740 42
    motor own_damage frequency   loss date 2023-01 0.918     0.006 10.021 42

S1 flagged policy series (5 of 119)
     line peril measure                          basis     brk  step  trend_pa   dbic  n
    motor     -  policy               mean vehicle_age 2024-02 0.789     0.108 24.583 48
 contents     -  policy share renewal_decision=renewed 2022-01 0.808     0.022 24.278 48
buildings     -  policy share renewal_decision=renewed 2022-01 0.812     0.020 24.168 48
    motor     -  policy share renewal_decision=renewed 2022-01 0.804     0.025 22.663 48

In [9]:
# 16b: follow-up to the step scan. Run in the same kernel right after 16. About 1-2 minutes.
# Purpose
#  A. Power: monthly series with 11 month-of-year effects missed the known BI step (dBIC 4.8 < 10) and the
#     third-party property step. Re-run the claim side on QUARTERLY series (3 season effects) and check the
#     positive controls again. This is a fix of the tool after a failed control, recorded as such.
#  B. Motor policy mix: vehicle_age and ncb_years both drop from Feb 2024 policies. Is it new business, a
#     change of definition, or does it continue into 2025 (test)? And does the claim relationship hold?
QTRS = pd.period_range("2021Q1", END.asfreq("Q") - (0 if END.month % 3 == 0 else 1), freq="Q")


def step_test_q(y, w, seasonal):
    """Same test on quarterly data: month-of-year effects replaced by quarter effects."""
    y = np.asarray(y, float); T = len(y); t = np.arange(T)
    w = np.nan_to_num(np.asarray(w, float)); ok = np.isfinite(y) & (y > 0) & (w > 0)
    ly = np.log(np.where(ok, y, 1.0))
    cols = [np.ones(T), t / 4] + ([(t % 4 == j).astype(float) for j in range(1, 4)] if seasonal else [])
    X0 = np.column_stack(cols)

    def fit(X):
        sw = np.sqrt(w[ok]); b, *_ = np.linalg.lstsq(X[ok] * sw[:, None], ly[ok] * sw, rcond=None)
        return max((((ly[ok] - X[ok] @ b) * sw) ** 2).sum(), 1e-12), b

    s0, _ = fit(X0); best = None
    for k in range(3, T - 3 + 1):
        s1, b1 = fit(np.column_stack([X0, (t >= k).astype(float)]))
        if best is None or s1 < best[0]:
            best = (s1, k, b1)
    n = ok.sum()
    return {"brk": str(QTRS[best[1]]), "step": float(np.exp(best[2][-1])),
            "trend_pa": float(np.exp(best[2][1]) - 1), "dbic": float(n * np.log(s0 / best[0]) - 2 * np.log(n))}


qrows = []
for line in LINES:
    p = load_policies(line, "train")
    expo_m = monthly_exposure(p, MONTHS)
    expo = expo_m.groupby(expo_m.index.asfreq("Q")).sum().reindex(QTRS)
    m = match_claims(claims_all, p, line, True, cap=None)
    m = m.assign(lq=m.loss_date.dt.to_period("Q"), rq=m.report_date.dt.to_period("Q"),
                 pq=m.period_start.dt.to_period("Q"))
    mat = m[(m.report_lag <= L) & (m.lq >= QTRS[0]) & (m.lq <= QTRS[-1]) & (m.status != "repudiated")]
    for per, g in mat.groupby("peril"):
        cnt = g.groupby("lq").size().reindex(QTRS, fill_value=0)
        if cnt.median() < 3 * MIN_COUNT:
            continue
        qrows.append({"line": line, "peril": per, "measure": "frequency", "basis": "loss date",
                      **step_test_q(cnt / expo, cnt, True)})
        cap = g.incurred_to_date.quantile(0.995)
        s = g[g.incurred_to_date > 0].assign(sev=lambda d: d.incurred_to_date.clip(upper=cap))
        for basis, col in [("loss date", "lq"), ("report date", "rq"), ("policy start", "pq")]:
            a = s.groupby(col).sev.agg(["mean", "size"]).reindex(QTRS)
            qrows.append({"line": line, "peril": per, "measure": "severity", "basis": basis,
                          **step_test_q(a["mean"], a["size"], False)})
        a = s[s.status == "closed"].groupby("lq").sev.agg(["mean", "size"]).reindex(QTRS)
        qrows.append({"line": line, "peril": per, "measure": "severity (closed)", "basis": "loss date",
                      **step_test_q(a["mean"], a["size"], False)})
q = pd.DataFrame(qrows); q.to_csv(OUT4 / "16b_quarterly.csv", index=False)
print("A. Quarterly scan — positive controls")
print(q[((q.peril == "bodily_injury") & q.measure.str.startswith("severity")) |
        ((q.peril.isin(["own_damage", "third_party_prop"])) & (q.measure == "frequency"))]
      .round(3).to_string(index=False))
fq = q[(q.dbic >= 10) & ((q.step - 1).abs() >= 0.05)].sort_values("dbic", ascending=False)
print(f"\nA. Quarterly scan — flagged ({len(fq)} of {len(q)})")
print(fq.round(3).to_string(index=False))

print("\nB. Motor policy mix by policy-start month (train 2023-07 .. 2024-12, then test 2025)")
pmo, pte = load_policies("motor", "train"), load_policies("motor", "test")
both = pd.concat([pmo, pte])
both["m"] = both.period_start.dt.to_period("M")
cols = [c for c in ["vehicle_age", "ncb_years", "new_business", "renewal_count", "driver_age"] if c in both]
tab = both.groupby("m")[cols].mean()
print(tab.loc["2023-07":].round(2).to_string())
print("\nB. vehicle_age distribution: policies starting 2023 vs 2024-02+ vs 2025 (share by age)")
grp = np.select([both.period_start < "2024-01-01", both.period_start < "2025-01-01"], ["<=2023", "2024"], "2025")
print(pd.crosstab(both.vehicle_age.clip(upper=15), grp, normalize="columns").round(3).T.to_string())
print("\nB. Same vehicles? renewals only: vehicle_age change from the previous period of the same policy")
r = both.sort_values(["policy_id", "period_start"])
r["d_age"] = r.groupby("policy_id").vehicle_age.diff()
r["y"] = r.period_start.dt.year
print(r.dropna(subset=["d_age"]).groupby("y").d_age.value_counts(normalize=True).unstack().round(3)
      .iloc[:, :6].to_string())

A. Quarterly scan — positive controls
 line            peril           measure        basis    brk  step  trend_pa   dbic
motor    bodily_injury          severity    loss date 2023Q1 0.809     0.035  7.020
motor    bodily_injury          severity  report date 2023Q1 0.807     0.037  2.261
motor    bodily_injury          severity policy start 2022Q3 0.786     0.046  6.262
motor    bodily_injury severity (closed)    loss date 2023Q1 0.784     0.031 16.305
motor       own_damage         frequency    loss date 2023Q1 0.918     0.005 20.702
motor third_party_prop         frequency    loss date 2022Q1 1.059    -0.044 17.185

A. Quarterly scan — flagged (3 of 75)
 line            peril           measure     basis    brk  step  trend_pa   dbic
motor       own_damage         frequency loss date 2023Q1 0.918     0.005 20.702
motor third_party_prop         frequency loss date 2022Q1 1.059    -0.044 17.185
motor    bodily_injury severity (closed) loss date 2023Q1 0.784     0.031 16.305

B. Motor p

In [10]:
# 17: year effects after rating factors, renewal checks. Same kernel after 11_setup. About 6-8 minutes.
# Purpose
#  A. Frequency GLM by peril with a YEAR factor plus all rating features (Fable review B3/B4):
#     a year effect that remains after the features is a true trend or step; one that vanishes is mix.
#     Covers TPP (trend or step?), own damage, buildings fire, contents theft, and every other peril.
#  B. Validation A/E by new business, renewal count and past-claim level: is renewal handled correctly?
#  C. Customers who renew under a NEW policy_id lose their claim history in our feature. How many?
#  D. Weather as a year effect (motor): do weather relativities shrink when a year factor is added?
# Rules fixed before results (Oct 20):
#  RA classification of the adjusted year index (2021 = 1):
#     "mix"   if every year is within +-3% of 2021-22 average  -> no adjustment;
#     "step"  if 2023 and 2024 are both beyond 3% in the same direction and differ from each other by < 2%;
#     "trend" if the fitted annual change is beyond +-2% and 2024 is further than 2023 from 2021-22.
#     A change of classification for an adjusted peril (own damage, TPP) or a trend in fire / theft is a
#     candidate for the target, tested in validation before adoption.
#  RB a segment (new business, renewal band, past claims) with A/E outside 0.90-1.10 in BOTH years in the
#     same direction is a model gap to investigate.
#  RC if > 2% of first periods are continuing customers under a new policy_id, history is linked by
#     customer and line instead of policy_id (tested before adoption).
#  RD the hypothesis "yearly weather acts as a year effect" is supported if the spread of weather
#     relativities falls by more than a third when a year factor is added.
from src.target import completeness, match_claims

EXTRA_DROP = {"motor": ["soil_type", "roof_type", "energy_rating", "building_age", "property_type", "bedrooms",
                        "flood_zone", "elevation_m", "distance_to_river_km", "discount_applied"],
              "buildings": ["discount_applied"], "contents": ["discount_applied"]}


def year_index(glm):
    r = glm.relativities()
    r = r[r.feature == "year_f"].set_index("level").relativity
    r = r.reindex(["2021", "2022", "2023", "2024"])
    return r / r["2021"]


def classify(ix):
    base = (ix["2021"] + ix["2022"]) / 2
    rel = ix / base
    slope = np.exp(np.polyfit(np.arange(4), np.log(ix.to_numpy()), 1)[0]) - 1
    d23, d24 = rel["2023"] - 1, rel["2024"] - 1
    if all(abs(rel - 1) <= 0.03):
        kind = "mix / none"
    elif abs(d23) > 0.03 and abs(d24) > 0.03 and np.sign(d23) == np.sign(d24) and abs(d24 - d23) < 0.02:
        kind = "step"
    elif abs(slope) > 0.02 and abs(d24) > abs(d23):
        kind = "trend"
    else:
        kind = "unclear"
    return kind, slope, rel


# ---------------- A ----------------
A = []
for line in LINES:
    p = load_policies(line, "train")
    df = data[line]["df"]
    feats = [c for c in feats_of(df, line) if c not in EXTRA_DROP[line]]
    m = match_claims(claims, p, line, True, cap=None)
    for per in sorted(m.peril.unique()):
        t0 = time.time()
        comp = {y: completeness(m, per, y) for y in [2021, 2022, 2023, 2024]}
        cnt = m[m.peril == per].groupby("row_id").size()
        d = df[["row_id", "year", "exposure_clean"] + feats].copy()
        d["n"] = d.row_id.map(cnt).fillna(0.0)
        d["w"] = d.exposure_clean * d.year.map(comp)
        d["year_f"] = d.year.astype(str)
        raw = d.groupby("year").n.sum() / d.groupby("year").w.sum()
        raw_ix = (raw / raw.iloc[0]); raw_ix.index = raw_ix.index.astype(str)
        glm = BandedTweedieGLM(power=1.0, alpha=1e-3, n_bands=GLM_CONFIG[line]["n_bands"], drop=None) \
            .fit(d[feats + ["year_f"]], d.n / d.w, d.w)
        adj = year_index(glm)
        kind, slope, rel = classify(adj)
        A.append({"line": line, "peril": per, "claims": int(d.n.sum()),
                  **{f"raw_{y}": raw_ix[y] for y in ["2022", "2023", "2024"]},
                  **{f"adj_{y}": adj[y] for y in ["2022", "2023", "2024"]},
                  "adj_trend_pa": slope, "class": kind})
        log(f"17A {line} {per}: {time.time() - t0:.0f}s -> {kind}")
A = pd.DataFrame(A); A.to_csv("../results/oct20/17A_year_effects.csv", index=False)
print("\nA. Frequency index by year (2021 = 1): raw vs after rating features (adj), and classification")
print(A.round(3).to_string(index=False))

# ---------------- B ----------------
B = []
for line in LINES:
    df = data[line]["df"]
    feats = [c for c in feats_of(df, line) if c not in EXTRA_DROP[line]]
    for vy, tys in FOLDS:
        tr, va = df[df.year.isin(tys)], df[df.year == vy].copy()
        glm = BandedTweedieGLM(**GLM_CONFIG[line]).fit(tr[feats], tr.target / tr.weight, tr.weight)
        va["pred"] = glm.predict_rate(va[feats]) * va.weight.to_numpy()
        va["renewal_band"] = pd.cut(va.renewal_count, [-1, 0, 1, 2, 4, 99], labels=["0", "1", "2", "3-4", "5+"])
        for seg in ["new_business", "renewal_band", "hist_1y"]:
            g = va.assign(s=va[seg].astype(str).fillna("unknown")).groupby("s", observed=True)[["target", "pred", "weight"]].sum()
            for lev, r in g.iterrows():
                B.append({"line": line, "segment": seg, "level": lev, "year": vy,
                          "A/E": r.target / r.pred, "weight_share": r.weight / g.weight.sum()})
B = pd.DataFrame(B)
bt = B.pivot_table(index=["line", "segment", "level"], columns="year", values=["A/E", "weight_share"]).round(3)
print("\nB. Validation A/E by renewal segment (GLM per line)")
print(bt.to_string())
ae = B.pivot_table(index=["line", "segment", "level"], columns="year", values="A/E")
gap = ae[((ae[2023] > 1.10) & (ae[2024] > 1.10)) | ((ae[2023] < 0.90) & (ae[2024] < 0.90))]
print(f"RB segments outside 0.90-1.10 in both years, same direction: {len(gap)}")
print(gap.round(3).to_string() if len(gap) else "  none")

# ---------------- C ----------------
print("\nC. First periods of a policy_id that continue a policy of the same customer and line (gap <= 31 days)")
for line in LINES:
    pp = pd.concat([load_policies(line, "train"), load_policies(line, "test")])
    first = pp.sort_values("period_start").groupby("policy_id").head(1)
    first = first[first.period_start > pd.Timestamp("2021-01-31")]
    mm = first[["row_id", "customer_id", "policy_id", "period_start", "new_business"]].merge(
        pp[["customer_id", "policy_id", "period_end"]].rename(columns={"policy_id": "pid2"}), on="customer_id")
    mm = mm[mm.pid2 != mm.policy_id]
    gapd = (mm.period_start - mm.period_end).dt.days
    cont = mm[(gapd >= 1) & (gapd <= 31)].drop_duplicates("row_id")
    print(f"  {line}: {len(cont):,} of {len(first):,} first periods ({len(cont) / len(first):.2%}); "
          f"{len(cont) / len(pp):.2%} of all rows; flagged new_business among them {cont.new_business.mean():.0%}")

# ---------------- D ----------------
print("\nD. Motor: spread (max / min) of weather relativities without and with a year factor")
line = "motor"; df = data[line]["df"].copy(); df["year_f"] = df.year.astype(str)
feats = [c for c in feats_of(df, line) if c not in EXTRA_DROP[line] and c != "year_f"]
wc = [c for c in WEATHER_COLS if c in feats] if "WEATHER_COLS" in dir() else \
     [c for c in ["hail_days", "wind_exposure_index", "freezing_days", "dry_summer_index", "wildfire_index",
                  "rainfall_mm", "mean_temperature_c"] if c in feats]
out = {}
for name, f in [("no year", feats), ("with year", feats + ["year_f"])]:
    g = BandedTweedieGLM(**GLM_CONFIG[line]).fit(df[f], df.target / df.weight, df.weight)
    r = g.relativities(); r = r[r.feature.isin(wc)]
    out[name] = r.groupby("feature").relativity.agg(lambda x: x.max() / x.min())
    if name == "with year":
        print("  year relativities:", (lambda s: (s / s.iloc[0]).round(3).to_dict())(
            g.relativities().query("feature == 'year_f'").set_index("level").relativity))
dd = pd.DataFrame(out); dd["shrink"] = 1 - (dd["with year"] - 1) / (dd["no year"] - 1)
print(dd.round(3).to_string())
print(f"RD average shrink of the weather spread: {dd.shrink.mean():.0%} -> "
      f"{'supported' if dd.shrink.mean() > 1 / 3 else 'not supported'}")

16:10:02 17A motor bodily_injury: 30s -> unclear
16:10:31 17A motor glass_hail: 29s -> step
16:11:03 17A motor own_damage: 32s -> unclear
16:11:29 17A motor theft: 26s -> unclear
16:12:01 17A motor third_party_prop: 32s -> mix / none
16:12:20 17A buildings accidental_damage: 20s -> unclear
16:12:37 17A buildings escape_of_water: 17s -> unclear
16:12:52 17A buildings fire: 15s -> unclear
16:13:14 17A buildings flood: 22s -> mix / none
16:13:32 17A buildings storm: 18s -> unclear
16:13:49 17A buildings subsidence: 18s -> trend
16:14:11 17A contents accidental_damage: 21s -> step
16:14:29 17A contents escape_of_water: 18s -> unclear
16:14:50 17A contents flood: 20s -> unclear
16:15:08 17A contents theft: 19s -> step

A. Frequency index by year (2021 = 1): raw vs after rating features (adj), and classification
     line             peril  claims  raw_2022  raw_2023  raw_2024  adj_2022  adj_2023  adj_2024  adj_trend_pa      class
    motor     bodily_injury    7512     0.946     0.955     0

In [12]:
# 18 E6: time effect in the GLM (motor, buildings). Same kernel after 11_setup. About 8-10 minutes.
# Purpose
#  Notebook 17 showed that, after rating factors, risk rises about 1-2.5% a year in almost every peril,
#  while raw frequency is flat: features that drift with time (NCB, vehicle age, renewal count, ...) make
#  the portfolio look safer each year, but losses did not fall. A model without time then predicts 2025 too low.
#  1. Mix decomposition: which features drive the predicted "safer each year" effect?
#  2. Variants, validated out of time (v8 basis: average weather):
#       a  current GLM (no time)
#       b1 + year factor; predict at the LAST training year's level
#       b2 + year factor; extrapolate a log-linear trend fitted to the training-year effects
#       c1/c2 (motor) as b1/b2, with the own damage / TPP frequency shifts (0.93 / 0.95) removed from the
#          target, since notebook 17 found their raw drop is mix (would be counted twice)
#  3. Effect on the 2025 prediction (totals vs a).
# Rules fixed before results (Oct 20):
#  T1 a variant replaces a if average D2 >= +0.0005, tails within 0.1 of a, and A/E moves toward 1 (or stays
#     within 0.02) in both years. Among passing variants, b1 (latest level) is preferred to b2 (extrapolated)
#     unless b2 is better by >= 0.0005.
#  Caveat: in validation the last training year (2022 or 2023) is mature; in the final model it is 2024,
#     which is the least mature (BI 56% reported, open reserves). The 2025 totals are therefore also shown
#     at the 2023 level; which level to use is a team decision recorded with both numbers.
#  T2 removing the frequency shifts (c) is adopted with the time term if it is not worse than the matching b
#     variant by more than 0.0005 (a simpler target). Validation actuals are identical: the shifts apply to
#     2021-22 losses only.
from src.target import ADJ, build_model_target

EXTRA_DROP = {"motor": ["soil_type", "roof_type", "energy_rating", "building_age", "property_type", "bedrooms",
                        "flood_zone", "elevation_m", "distance_to_river_km", "discount_applied"],
              "buildings": ["discount_applied"]}


def contributions(glm, X):
    """Log-contribution of each feature to each row's prediction."""
    Z = glm.enc_.transform(glm._band(X)).tocsc()
    names = glm.enc_.get_feature_names_out()
    fs = sorted(glm.enc_.feature_names_in_, key=len, reverse=True)
    owner = np.array([next(f for f in fs if n.startswith(f + "_")) for n in names])
    return pd.DataFrame({f: Z[:, np.where(owner == f)[0]] @ glm.model_.coef_[np.where(owner == f)[0]]
                         for f in glm.enc_.feature_names_in_}, index=X.index)


def year_rel(glm, years):
    r = glm.relativities()
    r = r[r.feature == "year_f"].set_index("level").relativity
    return r.reindex([str(y) for y in years])


def trend_from(rel):
    x = np.array([int(i) for i in rel.index]); return float(np.polyfit(x, np.log(rel.to_numpy()), 1)[0])


res, totals = [], {}
for line in ["motor", "buildings"]:
    t_line = time.time()
    p, pt = load_policies(line, "train"), load_policies(line, "test")
    F = add_history(build_features(line, "train", policies=p, weather_mode="average"), p, [p, pt], claims)
    Ft = add_history(build_features(line, "test", policies=pt, weather_mode="average"), pt, [p, pt], claims)
    feats = [c for c in feats_of(data[line]["df"], line) if c not in EXTRA_DROP[line]]
    targets = {"base": data[line]["df"][["row_id", "year", "exposure_clean", "target", "weight"]]}
    if line == "motor":
        mt, _, _ = build_model_target(claims, p, line, adj=dict(ADJ[line], freq_shift={}))
        targets["no_shift"] = mt
    T = {k: v.merge(F, on="row_id", how="left", validate="one_to_one").assign(year_f=lambda d: d.year.astype(str))
         for k, v in targets.items()}
    te_expo = pt.set_index("row_id").loc[Ft.row_id, "exposure_clean"].to_numpy()
    Ft = Ft.assign(year_f="2024")

    # 1. mix decomposition (current model, all years)
    d = T["base"]
    g0 = BandedTweedieGLM(**GLM_CONFIG[line]).fit(d[feats], d.target / d.weight, d.weight)
    C = contributions(g0, d[feats])
    w = d.weight.to_numpy()
    mix = pd.DataFrame({f: pd.Series(np.exp(C[f].to_numpy()) * w).groupby(d.year.to_numpy()).sum()
                        / pd.Series(w).groupby(d.year.to_numpy()).sum() for f in C.columns})
    mix = mix / mix.iloc[0]
    pred = pd.Series(g0.predict_rate(d[feats]) * w).groupby(d.year.to_numpy()).sum() / pd.Series(w).groupby(d.year.to_numpy()).sum()
    act = d.groupby("year").target.sum() / d.groupby("year").weight.sum()
    print(f"\n===== {line}: predicted mix vs actual by year (2021 = 1)")
    print(pd.DataFrame({"predicted (all features)": pred / pred.iloc[0], "actual (target per weight)": act / act.iloc[0]})
          .round(3).T.to_string())
    top = mix.loc[2024].sub(1).abs().sort_values(ascending=False).index[:8]
    print(f"features moving the predicted level most (index of their contribution, 2021 = 1):")
    print(mix[top].round(3).T.to_string())

    # 2. validation
    for vy, tys in FOLDS:
        last = max(tys)
        va0 = T["base"][T["base"].year == vy]
        tr = T["base"][T["base"].year.isin(tys)]
        ga = BandedTweedieGLM(**GLM_CONFIG[line]).fit(tr[feats], tr.target / tr.weight, tr.weight)
        res.append({"line": line, "year": vy, "variant": "a  current",
                    **metrics(va0, ga.predict_rate(va0[feats]) * va0.weight.to_numpy())})
        for key, lab in [("base", "b"), ("no_shift", "c")]:
            if key not in T:
                continue
            tr = T[key][T[key].year.isin(tys)]
            va = T[key][T[key].year == vy]
            assert np.isclose(va.target.sum(), va0.target.sum()), "validation actuals must be identical"
            gb = BandedTweedieGLM(**GLM_CONFIG[line]).fit(tr[feats + ["year_f"]], tr.target / tr.weight, tr.weight)
            p_last = gb.predict_rate(va[feats].assign(year_f=str(last))) * va.weight.to_numpy()
            rel = year_rel(gb, tys); g = trend_from(rel)
            res.append({"line": line, "year": vy, "variant": f"{lab}1 latest level", **metrics(va, p_last)})
            res.append({"line": line, "year": vy, "variant": f"{lab}2 trend {np.exp(g) - 1:+.1%}/yr",
                        **metrics(va, p_last * np.exp(g * (vy - last)))})
        log(f"18 {line} {vy} done")

    # 3. 2025 totals
    da = T["base"]
    ga = g0
    totals[(line, "a  current")] = float((ga.predict_rate(Ft[feats]) * te_expo).sum())
    for key, lab in [("base", "b"), ("no_shift", "c")]:
        if key not in T:
            continue
        dk = T[key]
        gb = BandedTweedieGLM(**GLM_CONFIG[line]).fit(dk[feats + ["year_f"]], dk.target / dk.weight, dk.weight)
        rel = year_rel(gb, [2021, 2022, 2023, 2024]); g = trend_from(rel)
        p_last = gb.predict_rate(Ft[feats + ["year_f"]]) * te_expo
        print(f"  {line} {key}: year relativities (2021 = 1) {(rel / rel.iloc[0]).round(3).to_dict()}; trend {np.exp(g) - 1:+.2%}/yr")
        totals[(line, f"{lab}1 latest level")] = float(p_last.sum())
        totals[(line, f"{lab}1 at 2023 level")] = float((gb.predict_rate(Ft[feats + ["year_f"]].assign(year_f="2023"))
                                                         * te_expo).sum())
        totals[(line, f"{lab}2 trend")] = float(p_last.sum() * np.exp(g))
    log(f"18 {line}: {time.time() - t_line:.0f}s")

r = pd.DataFrame(res); r.to_csv("../results/oct20/18_e6_time.csv", index=False)
for line in ["motor", "buildings"]:
    t = r[r.line == line].pivot(index="variant", columns="year", values=["D2", "A/E", "low", "top"])
    t.columns = [f"{a}_{b}" for a, b in t.columns]
    t["D2_avg"] = t[["D2_2023", "D2_2024"]].mean(axis=1)
    print(f"\n===== {line}: validation")
    print(t.round(4).to_string())
    base = t.loc["a  current"]
    for v in t.index.drop("a  current"):
        x = t.loc[v]
        gain = x.D2_avg - base.D2_avg
        tails = all(abs(x[f"{m}_{y}"] - 1) <= abs(base[f"{m}_{y}"] - 1) + 0.1 for y in [2023, 2024] for m in ["low", "top"])
        ae = all(abs(x[f"A/E_{y}"] - 1) <= abs(base[f"A/E_{y}"] - 1) or abs(x[f"A/E_{y}"] - 1) <= 0.02 for y in [2023, 2024])
        print(f"  T1 {v:22s} gain {gain:+.4f}  tails {'ok' if tails else 'FAIL'}  A/E {'ok' if ae else 'FAIL'}"
              f"  -> {'passes' if gain >= 0.0005 and tails and ae else 'no'}")
tt = pd.Series(totals)
print("\n2025 prediction totals (before loading), relative to the current model")
for line in ["motor", "buildings"]:
    s = tt[line]; print(f"  {line}: " + "  ".join(f"{k}: {v / s['a  current'] - 1:+.2%}" for k, v in s.items()))


===== motor: predicted mix vs actual by year (2021 = 1)
                            2021   2022   2023   2024
predicted (all features)     1.0  0.972  0.975  0.958
actual (target per weight)   1.0  0.995  1.001  0.981
features moving the predicted level most (index of their contribution, 2021 = 1):
                2021   2022   2023   2024
ncb_years        1.0  0.981  0.970  0.960
hist_2y          1.0  1.000  1.035  1.035
vehicle_age      1.0  0.991  0.982  0.973
hist_1y          1.0  0.977  0.977  0.976
years_licensed   1.0  1.002  0.981  0.983
renewal_count    1.0  1.005  1.011  1.013
driver_age       1.0  1.004  1.007  1.008
cover_type       1.0  0.999  0.998  0.997
16:29:10 18 motor 2023 done
16:30:19 18 motor 2024 done
  motor base: year relativities (2021 = 1) {'2021': 1.0, '2022': 1.059, '2023': 1.109, '2024': 1.106}; trend +3.53%/yr
  motor no_shift: year relativities (2021 = 1) {'2021': 1.0, '2022': 1.062, '2023': 1.089, '2024': 1.086}; trend +2.76%/yr
16:31:28 18 motor: 234s

In [13]:
# 19: level reconciliation and the frequency shifts. Same kernel after 11_setup. About 6-8 minutes.
# Purpose
#  1. Level reconciliation (all lines): compare the model's 2025 total with model-free figures, and split
#     the gap into "level" and "mix". If the model's 2025 total is below the latest mature year's actual
#     level, the reason must be a mix effect the model believes in; we show how large it is.
#  2. Motor frequency shifts (0.93 own damage, 0.95 TPP on 2021-22 losses): notebook 17 found the raw drops are
#     largely mix, so the shifts may count mix twice. Test removing them WITHOUT a year term
#     (E6 tested removal only together with a year term).
# Rules fixed before results (Oct 20):
#  L1 the reconciliation is a check, not a model change. A gap between the model total and the model-free
#     total at the latest mature year (2023) of more than 3% is reported with its split into level and mix.
#  F1 removing a shift is a simplification: adopted if average D2 does not fall by more than 0.0005, tails stay
#     within 0.1 of the current model, and A/E in each year does not move away from 1 by more than 0.01.
#     If several variants pass, the one that removes the most assumptions is adopted.
#     Validation actuals are identical (the shifts apply to 2021-22 losses only).
from src.target import ADJ, YEARS, build_model_target

EXTRA_DROP = {"motor": ["soil_type", "roof_type", "energy_rating", "building_age", "property_type", "bedrooms",
                        "flood_zone", "elevation_m", "distance_to_river_km", "discount_applied"],
              "buildings": ["discount_applied"], "contents": ["discount_applied"]}
TABLES = {}
for line in LINES:
    p, pt = load_policies(line, "train"), load_policies(line, "test")
    F = add_history(build_features(line, "train", policies=p, weather_mode="average"), p, [p, pt], claims)
    Ft = add_history(build_features(line, "test", policies=pt, weather_mode="average"), pt, [p, pt], claims)
    TABLES[line] = (p, pt, F, Ft)
    log(f"19 features {line}")

# ---------------- 1. level reconciliation ----------------
print("1. Level reconciliation (capped cost per unit of weight, 2025 level; GLM, average weather)")
rec = []
for line in LINES:
    p, pt, F, Ft = TABLES[line]
    feats = [c for c in feats_of(data[line]["df"], line) if c not in EXTRA_DROP[line]]
    d = data[line]["df"][["row_id", "year", "exposure_clean", "target", "weight"]].merge(F, on="row_id", how="left")
    g = BandedTweedieGLM(**GLM_CONFIG[line]).fit(d[feats], d.target / d.weight, d.weight)
    d["pred"] = g.predict_rate(d[feats]) * d.weight
    by = d.groupby("year")[["target", "pred", "weight"]].sum()
    actual, fitted = by.target / by.weight, by.pred / by.weight
    te_expo = pt.set_index("row_id").loc[Ft.row_id, "exposure_clean"].to_numpy()
    rate25 = g.predict_rate(Ft[feats]); model_total = float((rate25 * te_expo).sum())
    pred25 = float(np.average(rate25, weights=te_expo))
    mix = pred25 / fitted[2023]
    expo25 = te_expo.sum()
    out = {"line": line, **{f"actual_{y}": actual[y] for y in YEARS}, **{f"fitted_{y}": fitted[y] for y in YEARS},
           "model_2025_rate": pred25, "mix_2025_vs_2023": mix,
           "model_total": model_total, "free_2023": actual[2023] * expo25,
           "free_2023_x_mix": actual[2023] * mix * expo25, "free_avg_22_23": actual[[2022, 2023]].mean() * expo25}
    rec.append(out)
    print(f"\n  {line}")
    print("   actual per weight :", " ".join(f"{y} {actual[y]:7.1f}" for y in YEARS))
    print("   model in-sample   :", " ".join(f"{y} {fitted[y]:7.1f}" for y in YEARS),
          "  A/E", " ".join(f"{actual[y] / fitted[y]:.3f}" for y in YEARS))
    print(f"   model 2025 rate {pred25:.1f}; mix effect 2025 vs 2023 {mix - 1:+.2%} (what the features say)")
    gap = model_total / out["free_2023"] - 1
    print(f"   2025 total: model {model_total:,.0f}; model-free at 2023 level {out['free_2023']:,.0f} ({gap:+.2%}); "
          f"at 2023 level x model mix {out['free_2023_x_mix']:,.0f} "
          f"({model_total / out['free_2023_x_mix'] - 1:+.2%}: the level part)")
    if abs(gap) > 0.03:
        print(f"   L1: gap {gap:+.2%} > 3%: level part {model_total / out['free_2023_x_mix'] - 1:+.2%}, mix part {mix - 1:+.2%}")
pd.DataFrame(rec).to_csv("../results/oct20/19_level_reconciliation.csv", index=False)

# ---------------- 2. motor frequency shifts ----------------
line = "motor"; p, pt, F, Ft = TABLES[line]
feats = [c for c in feats_of(data[line]["df"], line) if c not in EXTRA_DROP[line]]
te_expo = pt.set_index("row_id").loc[Ft.row_id, "exposure_clean"].to_numpy()
VAR = {"a current (0.93 / 0.95)": ADJ[line]["freq_shift"],
       "d1 no TPP shift": {"own_damage": 0.93},
       "d2 no own damage shift": {"third_party_prop": 0.95},
       "d3 no shifts": {}}
res, tot = [], {}
for name, fs in VAR.items():
    t0 = time.time()
    mt, _, _ = build_model_target(claims, p, line, adj=dict(ADJ[line], freq_shift=fs))
    d = mt.merge(F, on="row_id", how="left", validate="one_to_one")
    for vy, tys in FOLDS:
        tr, va = d[d.year.isin(tys)], d[d.year == vy]
        g = BandedTweedieGLM(**GLM_CONFIG[line]).fit(tr[feats], tr.target / tr.weight, tr.weight)
        res.append({"variant": name, "year": vy, "va_total": va.target.sum(),
                    **metrics(va, g.predict_rate(va[feats]) * va.weight.to_numpy())})
    g = BandedTweedieGLM(**GLM_CONFIG[line]).fit(d[feats], d.target / d.weight, d.weight)
    tot[name] = float((g.predict_rate(Ft[feats]) * te_expo).sum())
    lv = d.groupby("year").target.sum() / d.groupby("year").weight.sum()
    log(f"19 {name}: {time.time() - t0:.0f}s; levels {lv.round(1).to_dict()}")
r = pd.DataFrame(res)
assert r.groupby("year").va_total.nunique().eq(1).all(), "validation actuals must be identical"
r.to_csv("../results/oct20/19_freq_shift.csv", index=False)
t = r.pivot(index="variant", columns="year", values=["D2", "A/E", "low", "top"])
t.columns = [f"{a}_{b}" for a, b in t.columns]
t["D2_avg"] = t[["D2_2023", "D2_2024"]].mean(axis=1)
t["2025_total_vs_a"] = pd.Series(tot) / tot["a current (0.93 / 0.95)"] - 1
print("\n2. Motor frequency shifts: validation and 2025 total")
print(t.round(4).to_string())
b = t.loc["a current (0.93 / 0.95)"]
passing = []
for v in t.index.drop("a current (0.93 / 0.95)"):
    x = t.loc[v]
    d2 = x.D2_avg - b.D2_avg >= -0.0005
    tails = all(abs(x[f"{m}_{y}"] - 1) <= abs(b[f"{m}_{y}"] - 1) + 0.1 for y in [2023, 2024] for m in ["low", "top"])
    ae = all(abs(x[f"A/E_{y}"] - 1) <= abs(b[f"A/E_{y}"] - 1) + 0.01 for y in [2023, 2024])
    ok = d2 and tails and ae
    print(f"  F1 {v:24s} D2 {x.D2_avg - b.D2_avg:+.4f} tails {'ok' if tails else 'FAIL'} A/E {'ok' if ae else 'FAIL'} -> {'passes' if ok else 'no'}")
    if ok:
        passing.append(v)
removed = {"d1 no TPP shift": 1, "d2 no own damage shift": 1, "d3 no shifts": 2}
print("F1 adopted:", max(passing, key=lambda v: removed[v]) if passing else "none (keep 0.93 / 0.95)")

16:41:38 19 features motor
16:41:40 19 features buildings
16:41:43 19 features contents
1. Level reconciliation (capped cost per unit of weight, 2025 level; GLM, average weather)

  motor
   actual per weight : 2021   526.3 2022   523.8 2023   526.7 2024   516.2
   model in-sample   : 2021   535.9 2022   521.0 2023   522.6 2024   513.6   A/E 0.982 1.005 1.008 1.005
   model 2025 rate 505.6; mix effect 2025 vs 2023 -3.25% (what the features say)
   2025 total: model 101,108,310; model-free at 2023 level 105,330,856 (-4.01%); at 2023 level x model mix 101,905,266 (-0.78%: the level part)
   L1: gap -4.01% > 3%: level part -0.78%, mix part -3.25%

  buildings
   actual per weight : 2021   456.4 2022   452.9 2023   460.6 2024   488.5
   model in-sample   : 2021   458.0 2022   450.0 2023   472.5 2024   477.0   A/E 0.996 1.006 0.975 1.024
   model 2025 rate 483.6; mix effect 2025 vs 2023 +2.35% (what the features say)
   2025 total: model 72,658,221; model-free at 2023 level 69,198,708 (+5.0

In [14]:
# 20: v9 = v8 with the motor frequency shifts removed (notebook 19, rule F1). Same kernel, after 14 has run
# (uses build(), SUB and the functions defined there). About 4 minutes.
# The adjustment is passed explicitly, so this cell gives the same result before or after the edit of
# src/target.py; after the edit, a fresh run of 11_setup + 14 reproduces v9 from the code alone.
# Expected change, written before submitting (Oct 20):
#   validation: motor D2 +0.0002, A/E 2023 1.031 -> 1.000, 2024 1.007 -> 1.003; motor 2025 total +0.28%
#   leaderboard: -0.0002 to +0.0005 (within the +-0.001 noise range). Reason for v9: one assumption fewer,
#   level closer to the validation actuals; not expected to move the score materially.
from src.target import ADJ, build_model_target

p_m = load_policies("motor", "train")
mt, comp_m, _ = build_model_target(claims, p_m, "motor", adj=dict(ADJ["motor"], freq_shift={}))
old = data["motor"]["df"]
data["motor"]["df"] = mt.merge(old.drop(columns=["year", "exposure_clean", "target", "weight"]),
                               on="row_id", how="left", validate="one_to_one")
print(f"motor target total: {old.target.sum():,.0f} -> {data['motor']['df'].target.sum():,.0f} "
      f"({data['motor']['df'].target.sum() / old.target.sum() - 1:+.2%}); completeness {comp_m.round(3).to_dict()}")

v9 = build("v9 (v8 + no motor frequency shifts)", "average", ["contents"])
v9.to_csv(SUB / "v9.csv", index=False)
v8f = pd.read_csv(SUB / "v8.csv").merge(v9, on=id_col, suffixes=("_v8", "_v9"))
a, b = v8f[f"{val_col}_v8"], v8f[f"{val_col}_v9"]
print(f"v9 vs v8: total {b.sum() / a.sum() - 1:+.2%}; correlation {np.corrcoef(a, b)[0, 1]:.5f}")
print(f"saved {SUB / 'v9.csv'}")
data["motor"]["df"] = old          # restore the setup table for any later cell

motor target total: 390,123,638 -> 396,380,969 (+1.60%); completeness {2021: 1.0, 2022: 0.998, 2023: 0.978, 2024: 0.814}

v9 (v8 + no motor frequency shifts)
  motor: GLM, k=1.0188, loading factor 1.0053; mean rate 509.70, total 101,930,661
  buildings: GLM, k=1.0407, flat loading 2.33; mean rate 485.93, total 73,007,884
  contents: LightGBM, 3 seeds; mean rate 108.63, total 16,308,227
v9 vs v8: total +0.14%; correlation 0.99998
saved ../submissions/v9.csv
